In [ ]:
# | default_exp features.ocf_offtarget

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class OCFOfftargetEvaluator(FeatureEvaluator):
    """Extracts off-target OCF metrics per tissue with cross-tissue aggregates.

    Cross-tissue derived metrics:
    - max_ocf_z: highest z-score across all tissues
    - n_tissues_elevated: count of tissues with z > 2.0
    - ocf_entropy: Shannon entropy of positive z-score distribution
    """

    name = "OCFOfftarget"
    source_file = ".OCF.offtarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "tissue" in cols:
                for row in df.to_dict("records"):
                    t = str(row["tissue"]).replace(" ", "_")
                    if "OCF" in cols and pd.notna(row["OCF"]):
                        extracted[f"{t}_OCF"] = float(row["OCF"])
                    if "ocf_z" in cols and pd.notna(row["ocf_z"]):
                        extracted[f"{t}_ocf_z"] = float(row["ocf_z"])

            # --- Cross-tissue aggregate metrics ---
            z_vals = {k: v for k, v in extracted.items() if k.endswith("_ocf_z")}
            if z_vals:
                z_arr = np.array(list(z_vals.values()))
                extracted["max_ocf_z_offtarget"] = float(np.max(z_arr))
                extracted["n_tissues_elevated_offtarget"] = float(np.sum(z_arr > 2.0))

                z_pos = z_arr[z_arr > 0]
                if len(z_pos) > 1:
                    p = z_pos / z_pos.sum()
                    extracted["ocf_entropy_offtarget"] = float(
                        -np.sum(p * np.log2(p + 1e-10))
                    )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = OCFOfftargetEvaluator()
df = pd.DataFrame([{"tissue": "Breast", "OCF": -13.0, "ocf_z": 1.2}])
res = evaluator.extract(df)
assert "Breast_OCF" in res